# byteSmart Basic Algorithms

This notebook keeps the work simple and beginner-friendly. It uses only the main data libraries plus the three required machine learning algorithms:

- `numpy`
- `pandas`
- `matplotlib`
- `LinearRegression`
- `KMeans`
- `LogisticRegression`

The goal is to understand the dataset with basic graphs and simple models, not to make the code overly complicated.

## 1. Setup

This cell imports the few libraries used in the notebook and connects Google Drive.

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive mount skipped:', e)

## 2. Load the Dataset

This cell finds the dataset zip file in Google Drive or in the Colab file area, then extracts the CSV files.

In [ ]:
ZIP_NAME = '14888121-20260708T173347Z-3-001.zip'

places_to_search = [Path('/content'), Path('/content/drive/MyDrive')]
zip_path = None

for place in places_to_search:
    if place.exists():
        matches = list(place.rglob(ZIP_NAME))
        if len(matches) > 0:
            zip_path = matches[0]
            break

if zip_path is None:
    raise FileNotFoundError('Upload the dataset zip file to Google Drive or to this Colab session.')

extract_dir = Path('/content/byteSmart_simple_data')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

data_dir = extract_dir / '14888121'
print('Using data folder:', data_dir)
print([file.name for file in data_dir.glob('*.csv')])

## 3. Clean the Data

The CSV files have a few extra rows at the top, so this cell cleans them and converts the important columns into numbers.

In [ ]:
def elapsed_to_hours(series):
    elapsed = pd.to_timedelta(series.astype(str), errors='coerce')
    return elapsed.dt.total_seconds() / 3600

def load_dry_ice(path):
    raw = pd.read_csv(path, header=None)
    dry = pd.DataFrame({
        'hours': pd.to_numeric(raw.iloc[4:, 2], errors='coerce'),
        'baseline_lb': pd.to_numeric(raw.iloc[4:, 5], errors='coerce'),
        'refrigerated_lb': pd.to_numeric(raw.iloc[4:, 6], errors='coerce')
    })
    return dry.dropna(subset=['hours'])

def load_test1(path):
    df = pd.read_csv(path, low_memory=False).iloc[2:].copy()
    df['hours'] = elapsed_to_hours(df['Time Elapsed'])

    for col in df.columns:
        if col not in ['date', 'time', 'Time Elapsed', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna(subset=['hours'])

def load_test2(path):
    df = pd.read_csv(path, low_memory=False).iloc[1:].copy()
    df['timestamp'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df['hours'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 3600

    for col in df.columns:
        if col not in ['TIMESTAMP', 'timestamp', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.dropna(subset=['hours'])

def temperature_columns(df, exclude):
    temp_cols = []
    for col in df.columns:
        if col not in exclude:
            if pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().sum() > 100:
                temp_cols.append(col)
    return temp_cols

dry = load_dry_ice(data_dir / 'Test1_DryIceWeight.csv')
test1 = load_test1(data_dir / 'Test1_TempCO2O2.csv')
test2 = load_test2(data_dir / 'Test2_TempCO2O2.csv')

test1_temp_cols = temperature_columns(
    test1,
    {'date', 'time', 'Time Elapsed', 'hours', 'O2', 'CO2', 'Ambient', 'Unnamed: 63', 'Unnamed: 64'}
)

test2_temp_cols = temperature_columns(
    test2,
    {'TIMESTAMP', 'timestamp', 'hours', 'O2', 'CO2'}
)

print('Dry ice rows:', len(dry))
print('Test 1 temperature sensors:', len(test1_temp_cols))
print('Test 2 temperature sensors:', len(test2_temp_cols))

## 4. Linear Regression: Dry Ice Mass Drop

Linear regression is used here to draw a straight trendline for dry ice mass over time. The slope tells us how quickly dry ice is being lost.

In [ ]:
plt.figure(figsize=(9, 5))
results = []

for column, label, color in [
    ('baseline_lb', 'Baseline', 'red'),
    ('refrigerated_lb', 'Refrigerated', 'blue')
]:
    data = dry[['hours', column]].dropna()

    x = data[['hours']]
    y = data[column]

    model = LinearRegression()
    model.fit(x, y)

    slope = model.coef_[0]
    results.append({
        'condition': label,
        'loss_rate_lb_per_hour': slope,
        'loss_rate_lb_per_day': slope * 24
    })

    plt.scatter(data['hours'], y, color=color, alpha=0.6, label=f'{label} data')
    plt.plot(data['hours'], model.predict(x), color=color, linewidth=2, label=f'{label} trendline')

plt.title('Dry Ice Mass Drop Over Time')
plt.xlabel('Elapsed time (hours)')
plt.ylabel('Dry ice mass (lb)')
plt.legend()
plt.show()

pd.DataFrame(results)

**What this shows:** The baseline line drops faster than the refrigerated line. This means the baseline condition lost dry ice faster, while the refrigerated condition kept the dry ice for a longer amount of time.

## 5. Linear Regression: Fastest-Warming Test 1 Sensors

This section uses a simple linear regression model for each Test 1 sensor. The sensor with the biggest positive slope warmed the fastest.

In [ ]:
warming_rates = []

for sensor in test1_temp_cols:
    sensor_data = test1[['hours', sensor]].dropna()

    if len(sensor_data) > 2:
        x = sensor_data[['hours']]
        y = sensor_data[sensor]

        model = LinearRegression()
        model.fit(x, y)

        warming_rates.append({
            'sensor': sensor,
            'warming_rate_F_per_hour': model.coef_[0]
        })

warming_rates = pd.DataFrame(warming_rates)
warming_rates = warming_rates.sort_values('warming_rate_F_per_hour', ascending=False)
fastest_sensors = warming_rates.head(10)

plt.figure(figsize=(9, 5))
plt.barh(fastest_sensors['sensor'], fastest_sensors['warming_rate_F_per_hour'], color='green')
plt.title('Test 1 Sensors That Warmed Fastest')
plt.xlabel('Warming rate (degrees F per hour)')
plt.ylabel('Sensor')
plt.gca().invert_yaxis()
plt.show()

fastest_sensors

**What this shows:** The top sensors warmed the fastest during Test 1. These are important because they show where the temperature increased the quickest.

## 6. Test 1 Sensor Spread Over Time

This graph shows the difference between the warmest and coldest Test 1 sensor at each time point.

In [ ]:
test1_spread = test1[test1_temp_cols].max(axis=1) - test1[test1_temp_cols].min(axis=1)

plt.figure(figsize=(10, 5))
plt.plot(test1['hours'], test1_spread, color='purple')
plt.title('Test 1 Sensor Spread Over Time')
plt.xlabel('Elapsed time (hours)')
plt.ylabel('Warmest sensor minus coldest sensor (degrees F)')
plt.show()

spread_summary = pd.DataFrame({
    'average_spread_F': [test1_spread.mean()],
    'median_spread_F': [test1_spread.median()],
    'maximum_spread_F': [test1_spread.max()]
})

spread_summary

**What this shows:** A bigger spread means the temperature was more uneven. Since the spread stays large for much of the test, Test 1 had very different temperatures depending on sensor location.

## 7. K-Means: Group Similar Test 1 Sensors

K-Means groups sensors that behave similarly. To keep this basic, each sensor is described using only its average temperature, standard deviation, minimum, and maximum.

In [ ]:
sensor_summary = pd.DataFrame({
    'sensor': test1_temp_cols,
    'average_temp_F': [test1[col].mean() for col in test1_temp_cols],
    'std_temp_F': [test1[col].std() for col in test1_temp_cols],
    'min_temp_F': [test1[col].min() for col in test1_temp_cols],
    'max_temp_F': [test1[col].max() for col in test1_temp_cols]
})

feature_cols = ['average_temp_F', 'std_temp_F', 'min_temp_F', 'max_temp_F']
X = sensor_summary[feature_cols]

# Simple scaling using pandas and numpy, so K-Means treats each column fairly.
X_scaled = (X - X.mean()) / X.std()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
sensor_summary['cluster'] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
for cluster in sorted(sensor_summary['cluster'].unique()):
    group = sensor_summary[sensor_summary['cluster'] == cluster]
    plt.scatter(group['average_temp_F'], group['std_temp_F'], label=f'Cluster {cluster}', s=70)

plt.title('K-Means Sensor Groups')
plt.xlabel('Average temperature (degrees F)')
plt.ylabel('Temperature variation')
plt.legend()
plt.show()

sensor_summary.groupby('cluster')[feature_cols].mean().round(2)

**What this shows:** Each dot is one sensor. Sensors close together had similar behavior. K-Means makes the large sensor dataset easier to understand by grouping similar sensors instead of checking every sensor one by one.

## 8. Logistic Regression: Categorize Test 1 vs Test 2

Logistic regression is used here to categorize whether a small time window looks like Test 1 or Test 2. The input features are simple: average temperature, temperature spread, average O2, and average CO2.

In [ ]:
def make_windows(df, temp_cols, label, window_size=120):
    simple = pd.DataFrame({
        'mean_temp_F': df[temp_cols].mean(axis=1),
        'spread_F': df[temp_cols].max(axis=1) - df[temp_cols].min(axis=1),
        'O2_mean': df['O2'],
        'CO2_mean': df['CO2']
    }).dropna()

    rows = []
    for start in range(0, len(simple) - window_size, window_size):
        window = simple.iloc[start:start + window_size]
        rows.append({
            'test_label': label,
            'mean_temp_F': window['mean_temp_F'].mean(),
            'spread_F': window['spread_F'].mean(),
            'O2_mean': window['O2_mean'].mean(),
            'CO2_mean': window['CO2_mean'].mean()
        })

    return pd.DataFrame(rows)

windows = pd.concat([
    make_windows(test1, test1_temp_cols, 'Test 1'),
    make_windows(test2, test2_temp_cols, 'Test 2')
], ignore_index=True)

feature_cols = ['mean_temp_F', 'spread_F', 'O2_mean', 'CO2_mean']

# Shuffle the rows using pandas, then make a simple 75% / 25% train-test split.
windows = windows.sample(frac=1, random_state=42).reset_index(drop=True)
split_index = int(len(windows) * 0.75)

train = windows.iloc[:split_index]
test = windows.iloc[split_index:]

X_train = train[feature_cols]
y_train = train['test_label']
X_test = test[feature_cols]
y_test = test['test_label']

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

predictions = log_model.predict(X_test)

accuracy = np.mean(predictions == y_test)
print('Accuracy:', round(accuracy, 3))

confusion = pd.crosstab(
    y_test,
    predictions,
    rownames=['Actual test'],
    colnames=['Predicted test']
)

confusion

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(confusion, cmap='Blues')
plt.title('Logistic Regression: Actual vs Predicted Test')
plt.xlabel('Predicted test')
plt.ylabel('Actual test')
plt.xticks(range(len(confusion.columns)), confusion.columns)
plt.yticks(range(len(confusion.index)), confusion.index)
plt.colorbar(label='Number of windows')

for row in range(confusion.shape[0]):
    for col in range(confusion.shape[1]):
        plt.text(col, row, confusion.iloc[row, col], ha='center', va='center', color='black')

plt.show()

**What this shows:** The logistic regression model checks whether each window looks more like Test 1 or Test 2. If most numbers are on the diagonal of the confusion matrix, the model did well. This shows the two tests had different temperature and gas patterns.

## Final Summary

- Linear regression showed that baseline dry ice dropped faster than refrigerated dry ice.
- Linear regression also found the Test 1 sensors that warmed the fastest.
- The Test 1 spread graph showed that temperatures were uneven across different sensor locations.
- K-Means grouped similar sensors together to make the dataset easier to understand.
- Logistic regression categorized whether a window looked like Test 1 or Test 2 using basic temperature and gas features.